# 1. shared Gemini: individual-history similarity

Compare a candidate with the separate embeddings of the latest 10 past-click events. Retrieve the top 200 by maximum history cosine (`max_history`) or average of its top 3 history cosines (`topk_history`). Retain `mean_profile` for the original normalized-average baseline. Embeddings remain frozen; no training or business reranking is performed.


## Individual-history retrieval experiment

Default experiment: `max_history`, latest 10 clicks, 1,000 fixed validation requests. Next compare `topk_history` with `history_top_k=3` and the existing `mean_profile` baseline. Keep seed, embeddings, text variant and eligibility identical. Use a new run ID for every aggregation setting. Article embeddings are reused; history caches are separate for each run. These are aggregation experiments, with no transformer fine-tuning or business reranking. The calculation now compares each candidate to up to 10 vectors, so measure runtime on the sample before full validation. GPU and CPU paths implement the same aggregation.


# 2. Setup and run configuration

In Colab, enable a GPU if desired, mount Drive and upload the **new similarity-only `catalogue_pipeline.py`**. The upload option below accepts Colab's renamed filenames and explicitly imports the uploaded file, avoiding stale module copies. `DRIVE_ROOT` must point to the actual shared-folder shortcut in your account.

Start in audit mode. Switch to run mode after checking the paths and costs. Your run ID stays stable while switching phases; assign `RUN_ID = None` to start a new run before rerunning configuration. Completed runs can be reused without scoring again if their configuration is unchanged.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

EXPERIMENT_ID = '3.1_pretrained_similarity_gemini'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    FORCE_UPLOAD = True  # Prompt again when updating the helper.
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    if FORCE_UPLOAD or not (CODE_DIR / 'catalogue_pipeline.py').exists():
        uploaded = files.upload()
        matches = [name for name in uploaded
                   if Path(name).stem.startswith('catalogue_pipeline')
                   and Path(name).suffix == '.py']
        if len(matches) != 1:
            raise ValueError(f'Select exactly one catalogue_pipeline Python file; received {list(uploaded)}')
        (CODE_DIR / 'catalogue_pipeline.py').write_bytes(uploaded[matches[0]])
else:
    candidates = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in candidates if (p / 'catalogue_pipeline.py').exists()), None)
    if CODE_DIR is None:
        raise FileNotFoundError('Open from the repository root or this experiment folder.')

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy>=1.26', 'pandas>=2.2', 'pyarrow>=15',
                'scikit-learn>=1.4'], check=True)
import numpy as np
import pandas as pd

helper_path = CODE_DIR / 'catalogue_pipeline.py'
spec = importlib.util.spec_from_file_location('catalogue_pipeline', helper_path)
cp = importlib.util.module_from_spec(spec)
sys.modules['catalogue_pipeline'] = cp
spec.loader.exec_module(cp)
if getattr(cp, 'PIPELINE_VERSION', None) != 'similarity_individual_history_v6':
    raise RuntimeError('Upload the new similarity-only helper, not the archived hybrid version.')
print('Loaded:', cp.__file__)


Mounted at /content/drive


Saving catalogue_pipeline.py to catalogue_pipeline.py
Loaded: /content/Experiments/3.1_pretrained_similarity_gemini/catalogue_pipeline.py


In [2]:
OWNER = ''  # Your name for final handoff.
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
SEED = 42
RUN_PHASE = 'run'  # 'audit' or 'run'
LOAD_RUN = None  # Optional completed similarity-only persistent run to reproduce.
RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + f'_history_seed{SEED}'
WORK = CODE_DIR / 'outputs' / RUN_ID
PERSISTENT_ROOT = DRIVE_ROOT / 'Experiments'
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'owner': OWNER, 'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT), 'seed': SEED, 'scope': 'development',
    'input_version': None,
    'embedding_source': {'provider': 'Gemini', 'model': None, 'revision': None},
    'embedding_column': 'text_embedding', 'embedding_dimension': None,
    'history_limit': 10,
    'history_aggregation': 'max_history',  # Or 'topk_history' or baseline 'mean_profile'.
    'history_top_k': 3,  # Used only by topk_history; averages available events if fewer than 3.
    'validation_requests': None,
    'export_requests': {'train': 25, 'validation': None, 'test': None},
    'catalogue_batch_size': 4096,
    'similarity_device': 'auto',  # 'cpu', 'cuda', or automatic CUDA/CPU fallback.
    'similarity_min_gpu_batch': 256,
    'hash_inputs': False,
    'max_index_gb': 20, 'max_export_pairs': 2_150_000_000,
    'max_export_gb': 10, 'max_scoring_seconds': 3600,
    'persistent_root': str(PERSISTENT_ROOT),
    'paths': {},  # Override actual Drive input paths here if needed.
}
CONFIG['code_provenance'] = {
    'helper_sha256': hashlib.sha256(helper_path.read_bytes()).hexdigest(),
    'commit': None, 'uncommitted_changes': None,
}
try:
    CONFIG['code_provenance'].update(
        commit=subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip(),
        uncommitted_changes=bool(subprocess.run(['git', '-C', str(CODE_DIR), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout),
    )
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print('Drive root exists:', DRIVE_ROOT.exists())
print('Local run:', WORK)
print('No classifier training or checkpoint is required.')


Drive root exists: True
Local run: /content/Experiments/3.1_pretrained_similarity_gemini/outputs/20261105_182643_history_seed42
No classifier training or checkpoint is required.


# 3. Load and check shared data

Load the shared text embedding lookups, article IDs/publication times, fixed click histories, and original request/candidate keys. The engineered feature tables supply **only IDs, timestamps and genuine click labels for evaluation**. Feature vocabulary, metadata predictors and scaled/log variants are not required.

Article catalogue IDs and vector IDs are audited separately. Duplicate source keys, unreadable vectors, and temporal history violations raise errors. Test click labels, if present in an updated release, are ignored and are not loaded into the inference index; this policy is recorded in the audit. Original test impression IDs can repeat across users; the request key is `user_id:impression_id`.

Eligibility is unchanged: use the declared ordered catalogue union, retain the first snapshot per article ID, and require known publication time no later than the request. There is no extra age or access cutoff. Missing text vectors remain eligible with a documented zero score. Confirm the same catalogue/policy across the group before model comparison.


In [3]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Pipeline:', audit['pipeline_version'])
print('Compute:', audit['compute'])
print('Catalogue articles:', audit['catalogue_articles'])
print('Catalogue articles missing vectors:', audit['catalogue_articles_without_vectors'])
print('Embedding dimension:', audit['embedding_info']['dimension'])
display(pd.DataFrame(audit['request_counts']).T)


Similarity-only compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 63654912, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
Pipeline: similarity_individual_history_v6
Compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 63654912, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
Catalogue articles: 20738
Catalogue articles missing vectors: 18
Embedding dimension: 768


,candidate_rows,requests,split_start,source_has_clicked_column,outcome_policy,outcome_column_loaded,sampling_rule
train,2585747,232887,2023-05-18T07:00:01,True,observed_labels_for_evaluation,True,Smallest BLAKE2b-64(seed:split:behaviour_id); ...
validation,344666,29110,2023-05-25T07:00:15,True,observed_labels_for_evaluation,True,Smallest BLAKE2b-64(seed:split:behaviour_id); ...
test,348951,29142,2023-05-25T07:00:10,True,ignored_for_test_inference,False,Smallest BLAKE2b-64(seed:split:behaviour_id); ...


In [4]:
print('Planned development exports:')
display(pd.DataFrame(audit['planned_exports']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
print('Full-split costs (not the default export scope):')
display(pd.DataFrame(audit['full_export_costs']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
cp.check_cost(PREPARED)


Planned development exports:


,requests,eligible_pairs,estimated_csv_bytes,empty_pools
train,25,463808,282500,0
validation,29110,566218581,328943000,0
test,29142,566598922,329304600,0


Full-split costs (not the default export scope):


,requests,eligible_pairs,estimated_csv_bytes,empty_pools
train,232887,4319178324,2631623100,0
validation,29110,566218581,328943000,0
test,29142,566598922,329304600,0


{'pairs': 1133281311,
 'estimated_csv_bytes': 658530100,
 'local_free_bytes': 120169811968}

# 4. Prepare individual-history inputs

With `max_history`, score each candidate against every available article vector from the latest 10 past clicks and take the maximum cosine. With `topk_history`, take the mean of its largest 3 cosines (or all available events when fewer than 3 exist). With `mean_profile`, use the original normalized mean profile. Select the latest 10 events before dropping missing vectors. Repeated clicks remain repeated events. Histories must predate the split; no current click labels enter ranking. Missing candidate vectors or entirely unavailable histories receive zero. A matrix of separate interests can remain usable when opposing vectors would cancel in a mean profile.


In [5]:
print('Aggregation:', CONFIG['history_aggregation'])
print('History limit:', CONFIG['history_limit'])
print('Top-k history:', CONFIG['history_top_k'])


Aggregation: max_history
History limit: 10
Top-k history: 3


# 5. Define and train / load model

**There is nothing to train.** No epochs, alpha trials, classifier checkpoints, scalers or learned feature weights are used. This section saves a small JSON inference specification or loads the same specification from a completed similarity-only run. Old hybrid `selected.joblib` files are incompatible and are not read.

Set `RUN_PHASE='run'` after the audit. Rerun sections 3-4 if changing configuration. Use a fresh run ID for a different encoder, text variant or history setting. No `cp.train()` call is needed.


In [6]:
MODEL = None
if PREPARED['work'] != WORK or PREPARED['config'] != CONFIG:
    raise RuntimeError('Rerun sections 3-4 after changing configuration.')
if RUN_PHASE == 'run':
    MODEL = cp.load_run(PREPARED, LOAD_RUN) if LOAD_RUN else cp.define_model(PREPARED)
    print(json.dumps(MODEL, indent=2))
else:
    print('Audit only. Set RUN_PHASE="run" after reviewing costs.')


{
  "pipeline_version": "similarity_individual_history_v6",
  "method": "max_history_cosine",
  "history_aggregation": "max_history",
  "history_top_k": null,
  "history_limit": 10,
  "article_normalization": "L2 per nonmissing article vector",
  "user_profile": "maximum cosine over available selected past-click article vectors",
  "history_snapshot": "strictly before first request of each split; no behavior-log updates",
  "repeated_clicks": "retain repeated weight/events, including in top-k aggregation",
  "missing_vectors_or_history": "score 0; retain eligible articles; report flags and ties",
  "score_semantics": "aggregated cosine similarity in [-1,1], not a probability",
  "learned_parameters": false,
  "fine_tuned_encoder": false,
  "shortlist_size": 200,
  "tie_policy": "descending cosine, ascending original article ID"
}


# 6. Validation diagnostics and explainability

Evaluate the fixed validation-request sample against its **full eligible catalogue**. Primary metric: mean known-click Recall@200. Secondary: observed-click nDCG@200. Keep excluded original known clicks in both denominators/ideals. Empty pools with known clicks receive zero. Unshown outcomes remain unknown, not negative training labels.

There is no automatic tuning search in this notebook. If you later compare history settings, encoders or text variants, fix the same request IDs and select using validation Recall@200, then nDCG@200. Never select using test outcomes. Previously inspected/repeated validation is development evidence, not an untouched final test.


In [7]:
VALIDATION = None
if RUN_PHASE == 'run':
    VALIDATION = cp.evaluate(PREPARED, MODEL)
    display(pd.json_normalize(VALIDATION['metrics']))
    print('Scoring seconds:', round(VALIDATION['seconds'], 2))
    print('Catalogue pairs scored:', VALIDATION['eligible_pairs'])
    print('History coverage:', VALIDATION['history_coverage'])


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,eligible_only_recall@200,requests_with_ties_crossing_200,known_click_eligibility_coverage,ties,outcomes,requests_without_usable_user_profile,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.047774,0.008644,29110,0,0,0,29249,0.047774,6,1.0,"descending score, ascending original article I...",unshown outcomes unknown; zero observed gain i...,0,0,0,0


Scoring seconds: 183.95
Catalogue pairs scored: 566218581
History coverage: {'users_requested': 1719, 'users_found': 1719, 'missing_history_users': 0, 'events_missing_time_excluded': 0, 'selected_history_events': 16906, 'selected_events_with_vectors': 16906, 'history_limit': 10, 'history_aggregation': 'max_history', 'history_top_k': None}


Export `rankings/{split}/submission_top_200.parquet` with **behaviour_id,article_ids**: one request per row and a native integer list in descending recommendation order. Up to 200 IDs are retained; empty pools have an empty list. The CSV companion retains aligned similarity scores for diagnostics. No exhaustive catalogue-score file is saved.


In [8]:
RESULT = None
if RUN_PHASE == 'run':
    RESULT = cp.execute(PREPARED, MODEL)
    display(pd.json_normalize(RESULT['metrics']))
    print('Exported-validation metrics:')
    display(pd.json_normalize(RESULT['exported_validation']))
    print('Export checks:', RESULT['checks'])
    print('Complete final handoff:', RESULT['handoff_complete'])


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,eligible_only_recall@200,requests_with_ties_crossing_200,known_click_eligibility_coverage,ties,outcomes,requests_without_usable_user_profile,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.047774,0.008644,29110,0,0,0,29249,0.047774,6,1.0,"descending score, ascending original article I...",unshown outcomes unknown; zero observed gain i...,0,0,0,0


Exported-validation metrics:


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,known_click_eligibility_coverage,eligible_only_recall@200,requests_with_ties_crossing_200,source,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.047774,0.008644,29110,0,0,0,29249,1.0,0.047774,6,validated exported catalogue shortlists; origi...,0,0,0


Export checks: {'train': {'catalogue_pairs_rescored_for_verification': 463808, 'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'full_score_rows_exported': 0}, 'validation': {'catalogue_pairs_rescored_for_verification': 566218581, 'requests_verified': 29110, 'shortlisted_articles_verified': 5822000, 'full_score_rows_exported': 0}, 'test': {'catalogue_pairs_rescored_for_verification': 566598922, 'requests_verified': 29142, 'shortlisted_articles_verified': 5828400, 'full_score_rows_exported': 0}}
Complete final handoff: False


## Native-list Parquet submission

New runs automatically write the Parquet shortlist. For an already completed run, reload the updated Gemini helper and run just the next cell with WORK pointing at that run. It converts existing CSVs without embedding generation or catalogue scoring and copies the Parquet files/updated manifest into the existing persistent bundle. Keep the existing run ID for this format-only conversion.


In [9]:
completed_result = globals().get('RESULT')
if completed_result is None and (WORK / 'result.json').is_file():
    completed_result = json.loads((WORK / 'result.json').read_text())
persistent = completed_result['persistence']['persistent_location'] if completed_result and completed_result['persistence']['persistent_copy_complete'] else None
PARQUET_SUBMISSIONS = cp.export_parquet_submissions(WORK, persistent)
print(json.dumps(PARQUET_SUBMISSIONS, indent=2))
display(pd.read_parquet(WORK / 'rankings/validation/submission_top_200.parquet').head())


{
  "train": {
    "path": "/content/Experiments/3.1_pretrained_similarity_gemini/outputs/20261105_182643_history_seed42/rankings/train/submission_top_200.parquet",
    "rows": 25,
    "article_ids": 5000,
    "schema": "behaviour_id:string,article_ids:list<int64>",
    "order": "same recommendation order as the source CSV; no sorting/grouping applied"
  },
  "validation": {
    "path": "/content/Experiments/3.1_pretrained_similarity_gemini/outputs/20261105_182643_history_seed42/rankings/validation/submission_top_200.parquet",
    "rows": 29110,
    "article_ids": 5822000,
    "schema": "behaviour_id:string,article_ids:list<int64>",
    "order": "same recommendation order as the source CSV; no sorting/grouping applied"
  },
  "test": {
    "path": "/content/Experiments/3.1_pretrained_similarity_gemini/outputs/20261105_182643_history_seed42/rankings/test/submission_top_200.parquet",
    "rows": 29142,
    "article_ids": 5828400,
    "schema": "behaviour_id:string,article_ids:list<int64>

,behaviour_id,article_ids
0,393380:497194197,"[9778628, 9778804, 9779263, 9779577, 9780019, ..."
1,340393:212944368,"[9779507, 9779511, 9779520, 9779577, 9779580, ..."
2,1895052:36672692,"[9778843, 9778905, 9778917, 9778952, 9779417, ..."
3,450639:469050628,"[9777397, 9778915, 9779185, 9779263, 9779564, ..."
4,340393:212944369,"[9779507, 9779511, 9779520, 9779577, 9779580, ..."


# 8. Reload check

The full run bundle is copied to shared Drive, with the JSON inference model, code, source versions and a score probe. The helper reloads code/configuration from that persistent copy, reopens the versioned text embedding tables, recomputes probe cosine scores and checks agreement within 2e-6. No classifier is fitted or loaded. Temporary SQLite indexes are rebuilt from the recorded source files when needed rather than being part of the handoff model.

Partial/failed persistence is recorded as incomplete. Missing real-data access cannot be replaced by synthetic performance results. CUDA is optional and its hardware parity check is skipped locally when no CUDA hardware is available.


In [10]:
if RESULT is not None:
    print(json.dumps(RESULT['persistence'], indent=2))
    if RESULT['persistence']['persistent_copy_complete']:
        saved = Path(RESULT['persistence']['persistent_location'])
        print('Saved fixed inference model:', saved / 'model/model_config.json')
        print('No learned model weights/checkpoint are needed.')


{
  "persistent_copy_complete": true,
  "persistent_location": "/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.1_pretrained_similarity_gemini/20261105_182643_history_seed42",
  "copied_files_verified_by_size": 19,
  "reload_from_persistent_copy": true,
  "embedding_lookup_reopened": true,
  "learned_parameters": false,
  "rtol": 2e-06,
  "atol": 2e-06,
  "probe_rows": 10
}
Saved fixed inference model: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.1_pretrained_similarity_gemini/20261105_182643_history_seed42/model/model_config.json
No learned model weights/checkpoint are needed.


# 9. Business interpretation and handoff

Report whether direct pretrained content similarity recovers known clicked news in the catalogue top 200, the effect of unavailable profiles/vectors, observed-click ranking metrics, compute and storage cost. The Stage 2 owner receives the top-200 IDs plus the article/user information needed for the separate freshness/diversity/continuity re-ranker.

A top-200 similarity score is not click probability and does not establish retention gains. Do not compare these catalogue results directly with older recorded-slate hybrid results. The shared Drive data has not been run in the local coding environment; implementation verification uses synthetic fixtures only. Run the notebook on the actual shared files before reporting project metrics or a complete persistent handoff.


In [11]:
if RESULT is not None:
    example = json.loads((WORK / 'validation_example.json').read_text())
    display(pd.DataFrame(example['articles']))
    print('Validation request:', example['behaviour_id'])
    print(example['note'])
    print('Bundle:', RESULT['persistence'].get('persistent_location', str(WORK)))
    print('Test outcome metrics: unavailable.')


,article_id,cosine_similarity,candidate_vector_missing,user_profile_missing,history_vector_coverage
0,9778628,1.0,False,False,1.0
1,9778804,1.0,False,False,1.0
2,9779263,1.0,False,False,1.0
3,9779577,1.0,False,False,1.0
4,9780019,1.0,False,False,1.0


Validation request: 393380:497194197
Scores are direct similarity, with no learned weighting or click probability.
Bundle: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.1_pretrained_similarity_gemini/20261105_182643_history_seed42
Test outcome metrics: unavailable.
